# Sankey Diagram - Current System 2025 (BNA)

Resources -> conversion technologies -> end uses, for the **current system** scenario
(`norte_amazonia_reality_2025`), aggregated over the five cells C1-C5.

The flow table is built by the repository's own post-processing package:
`write_sankey_file` turns `outputs/regional_results/Year_balance.csv` into
`input2sankey_<cell>.csv`. The diagram itself is assembled below and displayed inline.

The grouping dictionaries of `output_to_sankey_csv` come from an older fork whose technology
set differs from this repository's; they are patched in memory only (section 1), so the
package on disk is left untouched.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

In [2]:
def find_repo_root():
    """Repository root, whether this notebook runs from scripts/ or from the root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'esmc').is_dir() and (candidate / 'case_studies').is_dir():
            return candidate
    raise RuntimeError(f'repository root not found above {here}')


ROOT       = find_repo_root()
SPACE_ID   = 'C1_C2_C3_C4_C5'
CASE_STUDY = 'norte_amazonia_reality_2025'   # current system 2025 (scenario CS)

REG = ROOT / 'case_studies' / SPACE_ID / CASE_STUDY / 'outputs' / 'regional_results'

# scripts/ is not the repository root, and another EnergyScope fork is pip-installed
# under the same package name: put THIS repository first on the import path.
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

WIDTH_PX, HEIGHT_PX = 630, 620   # height tuned so that no node label overlaps
FONT_SIZE = 13

assert REG.is_dir(), f'case study outputs not found: {REG}'
print(f'case study : {CASE_STUDY}')
print(f'inputs     : {REG.relative_to(ROOT)}')

case study : norte_amazonia_reality_2025
inputs     : case_studies\C1_C2_C3_C4_C5\norte_amazonia_reality_2025\outputs\regional_results


---
## 1. In-memory compatibility patch of the grouping dictionaries

`output_to_sankey_csv` carries hard-coded technology/layer groupings inherited from an
older fork. Three mismatches break it on this repository:

1. **Technologies that no longer exist** (`DEC_ELEC_COLD_FAN`, `GASIFICATION_SNG`,
   `PYROLYSIS_BIOWASTE_TO_FUELS`, `BIOETHANOL`, `BIODIESEL`) raise a `KeyError`.
2. **Active rows covered by no group are silently dropped** - in particular
   `ELECTRICITY` (the national grid import from the SIN), `LPG`, `LNG`,
   `REGASIFICATION` and `RES_SOLAR`. Without them the grid import and the LPG cooking
   chain would be missing from the diagram.
3. `StorageLayer` points at a layer name (`Heat LT DEC`) that the layer grouping calls
   `Heat LT`.

On top of the strict repair, the aggregated groups `Solar PV` and `Diesel Genset` are
**split** so that the four reference figures of the current system read directly off the
diagram: utility PV (Cobija) vs home-system PV, and grid gensets vs home-system gensets.

Everything below rebinds module attributes at runtime; the file on disk is untouched.


In [3]:
from esmc.postprocessing.draw_sankey import output_to_sankey_csv as sankey_mod
from esmc.postprocessing.draw_sankey.output_to_sankey_csv import write_sankey_file

# guard: the package must come from this repository, not from another installed fork
assert Path(sankey_mod.__file__).is_relative_to(ROOT), (
    f'esmc resolved to {sankey_mod.__file__} instead of {ROOT}')

year_balance = pd.read_csv(REG / 'Year_balance.csv', sep=';', index_col=[0, 1]).fillna(0)
sto_assets   = pd.read_csv(REG / 'Sto_assets.csv',   sep=';', index_col=[0, 1]).fillna(0)

TECHS  = set(year_balance.index.get_level_values(1))
LAYERS = set(year_balance.columns)
STOS   = set(sto_assets.index.get_level_values(1))

# split the aggregated PV / genset groups
sankey_mod.RegroupElements.pop('Solar PV', None)
sankey_mod.RegroupElements.pop('Diesel Genset', None)
sankey_mod.TechLayer.pop('Solar PV', None)
sankey_mod.TechLayer.pop('Diesel Genset', None)

# declare the groups missing for this technology set
sankey_mod.RegroupElements.update({
    'Utility PV':          ['PV_UTILITY', 'PV_ROOFTOP'],
    'Home-system PV':      ['PV_HS'],
    'Diesel gensets':      ['GENSET_DIESEL'],
    'Home diesel gensets': ['HS_DIESEL'],
    'Grid import (SIN)':   ['ELECTRICITY'],
    'LPG supply':          ['LPG'],
    'LNG supply':          ['LNG'],
    'Regasification':      ['REGASIFICATION'],
    'Solar':               ['RES_SOLAR'],
})
sankey_mod.TechLayer.update({
    'Utility PV':          'RES Solar',
    'Home-system PV':      'RES Solar',
    'Diesel gensets':      'Diesel',
    'Home diesel gensets': 'Diesel',
})
sankey_mod.RegroupLayers['LNG'] = ['LNG']
sankey_mod.LayerColor['LNG']    = '#B8860B'

# StorageLayer points at 'Heat LT DEC', which the layer grouping calls 'Heat LT'
sankey_mod.StorageLayer = {'DEC Sto.': ['Heat LT'], 'DHN Sto.': ['Heat LT DHN']}

# drop every entry absent from this run
def keep_existing(groups, available):
    """Filter a regrouping dict down to the elements present in this run."""
    filtered = {name: [e for e in members if e in available] for name, members in groups.items()}
    return {name: members for name, members in filtered.items() if members}

dropped = {n: [e for e in m if e not in TECHS] for n, m in sankey_mod.RegroupElements.items()}
dropped = {n: m for n, m in dropped.items() if m}

sankey_mod.RegroupElements = keep_existing(sankey_mod.RegroupElements, TECHS)
sankey_mod.RegroupLayers   = keep_existing(sankey_mod.RegroupLayers,   LAYERS)
sankey_mod.EndUseStorage   = keep_existing(sankey_mod.EndUseStorage,   STOS)

print('Technologies dropped (absent from this technology set):')
for name, members in dropped.items():
    print(f'  {name:22s} -> {members}')

# no active row may be left out of the diagram
totals   = year_balance.groupby(level=1).sum()
active   = totals[totals.abs().sum(axis=1) > 1e-6].index
grouped  = {t for members in sankey_mod.RegroupElements.values() for t in members}
orphans  = [t for t in active if t not in grouped]
print(f'\nActive rows not represented in the Sankey: {orphans if orphans else "none"}')

Technologies dropped (absent from this technology set):
  Cooling                -> ['DEC_ELEC_COLD_FAN']
  Gasifi SNG             -> ['GASIFICATION_SNG']
  Pyrolise               -> ['PYROLYSIS_BIOWASTE_TO_FUELS']
  Bioethanol imports     -> ['BIOETHANOL']
  Biodiesel imports      -> ['BIODIESEL']

Active rows not represented in the Sankey: none


---
## 2. Build the Sankey input file

`write_sankey_file` writes one `input2sankey_<cell>.csv` per cell plus an aggregated
`input2sankey_Total.csv` in `outputs/regional_results/`. The console trace below is the
package's own (Spanish) log; the "electrical losses" line is the difference between the
`END_USES` electricity row and the electricity demand declared in `reg_demands.dat`,
i.e. the model's network losses (`loss_network` = 15.96 % on the ELECTRICITY layer).


In [4]:
write_sankey_file(space_id=SPACE_ID, case_study=CASE_STUDY)

flows = pd.read_csv(REG / 'input2sankey_Total.csv')
flows = flows[flows['realValue'].abs() > 1e-9].reset_index(drop=True)   # drop empty links
print(f'\n{len(flows)} non-zero links written to {(REG / "input2sankey_Total.csv").relative_to(ROOT)}')
flows

Leyendo demandas eléctricas regionales...
Región C1: Demanda eléctrica = 2.933 TWh
Región C2: Demanda eléctrica = 0.084 TWh
Región C3: Demanda eléctrica = 26.541 TWh
Región C4: Demanda eléctrica = 7.090 TWh
Región C5: Demanda eléctrica = 18.061 TWh
Total del sistema: Demanda eléctrica = 54.709 TWh



--- Región C1 ---
Generación eléctrica total: 5.1 GWh
Demanda eléctrica real: 2.9 GWh
Pérdidas eléctricas: 2.2 GWh



--- Región C2 ---
Generación eléctrica total: 0.1 GWh
Demanda eléctrica real: 0.1 GWh
Pérdidas eléctricas: 0.0 GWh



--- Región C3 ---
Generación eléctrica total: 43.1 GWh
Demanda eléctrica real: 26.5 GWh
Pérdidas eléctricas: 16.6 GWh



--- Región C4 ---
Generación eléctrica total: 11.2 GWh
Demanda eléctrica real: 7.1 GWh
Pérdidas eléctricas: 4.1 GWh



--- Región C5 ---
Generación eléctrica total: 28.2 GWh
Demanda eléctrica real: 18.1 GWh
Pérdidas eléctricas: 10.1 GWh



--- Región Total ---
Generación eléctrica total: 87.7 GWh
Demanda eléctrica real: 54.7 GWh
Pérdidas eléctricas: 33.0 GWh

26 non-zero links written to case_studies\C1_C2_C3_C4_C5\norte_amazonia_reality_2025\outputs\regional_results\input2sankey_Total.csv


,source,target,realValue,layerID,layerColor,layerUnit
0,Home-system PV,Elec,0.370,RES Solar,#FF7F50,GWh
1,Diesel gensets,Elec,184.921,Diesel,#D3D3D3,GWh
2,Home diesel gensets,Elec,2.587,Diesel,#D3D3D3,GWh
3,Regasification,Fossil Gas,2.703,Fossil Gas,#FFD700,GWh
4,Grid import (SIN),Elec,13.857,Elec,#00BFFF,GWh
5,Prod. & Imp. Diesel,Diesel,511.859,Diesel,#D3D3D3,GWh
6,LPG supply,LPG,176.147,LPG,#da70d6,GWh
7,LNG supply,LNG,2.757,LNG,#B8860B,GWh
8,DEC Heat,Heat LT,5.101,Heat LT,#6a5acd,GWh
9,Stoves,Cooking,112.969,Cooking,#00CED1,GWh


---
## 3. Flow totals per layer - cross-check  [cross-check, not in the manuscript]

Three checks:

* **3.a** the four reference values of the current system, taken from `Year_balance.csv`
  (the authoritative source) and compared with the expected figures;
* **3.b** the total flow carried by each layer of the diagram;
* **3.c** closure of the electricity balance (supply = uses).


In [5]:
elec = year_balance.groupby(level=1).sum()['ELECTRICITY']

REFERENCE = [
    ('Diesel gensets (grid)',      ['GENSET_DIESEL'],       184.9),
    ('Utility PV (Cobija)',        ['PV_UTILITY'],            7.8),
    ('National grid import (SIN)', ['ELECTRICITY'],          13.9),
    ('Home systems (PV + genset)', ['PV_HS', 'HS_DIESEL'],    3.0),
]

check = pd.DataFrame(
    [{'Flow': label,
      'Year_balance rows': ' + '.join(rows),
      'Model [GWh]': elec.reindex(rows).sum(),
      'Expected [GWh]': expected}
     for label, rows, expected in REFERENCE]
)
check['Deviation [GWh]'] = check['Model [GWh]'] - check['Expected [GWh]']
check['Match'] = np.where(check['Deviation [GWh]'].abs() < 0.06, 'OK', 'CHECK')
print('3.a  Current-system reference flows (electricity layer)')
display(check.set_index('Flow'))

3.a  Current-system reference flows (electricity layer)


,Year_balance rows,Model [GWh],Expected [GWh],Deviation [GWh],Match
Flow,,,,,
Diesel gensets (grid),GENSET_DIESEL,184.921,184.900,0.021,OK
Utility PV (Cobija),PV_UTILITY,7.829,7.800,0.029,OK
National grid import (SIN),ELECTRICITY,13.857,13.900,-0.043,OK
Home systems (PV + genset),PV_HS + HS_DIESEL,2.957,3.000,-0.043,OK


In [6]:
by_layer = (flows.groupby('layerID')['realValue'].agg(['sum', 'count'])
                 .rename(columns={'sum': 'Total flow [GWh]', 'count': 'Links'})
                 .sort_values('Total flow [GWh]', ascending=False))
print('3.b  Total flow per layer (input2sankey_Total.csv)')
display(by_layer)

# tier decomposition: sources -> conversion -> sinks
sources = set(flows['source'])
targets = set(flows['target'])
tier_in  = sorted(sources - targets)     # primary supply and imports
tier_out = sorted(targets - sources)     # final uses

supply = (flows[flows['source'].isin(tier_in)].groupby('source')['realValue'].sum()
                .sort_values(ascending=False).rename('GWh'))
uses   = (flows[flows['target'].isin(tier_out)].groupby('target')['realValue'].sum()
                .sort_values(ascending=False).rename('GWh'))

print(f'\nPrimary supply and imports  (total {supply.sum():,.1f} GWh)')
display(supply.to_frame())
print(f'Final uses  (total {uses.sum():,.1f} GWh)')
display(uses.to_frame())

3.b  Total flow per layer (input2sankey_Total.csv)


,Total flow [GWh],Links
layerID,,
Diesel,"1,211.225",6
LPG,352.295,2
Elec,135.742,6
Cooking,112.969,1
Elec Demand,54.709,1
Elec_Losses,32.970,1
RES Solar,8.199,2
Solar,8.199,2
LNG,5.514,2



Primary supply and imports  (total 712.8 GWh)


,GWh
source,
Prod. & Imp. Diesel,511.859
LPG supply,176.147
Grid import (SIN),13.857
Solar,8.199
LNG supply,2.757


Final uses  (total 328.8 GWh)


,GWh
target,
Cooking,112.969
Elec Demand,54.709
Food preservation,49.802
Elec Losses,32.970
Cooling,32.933
Lighting,23.186
Mechanical Energy,17.082
Heat LT,5.101


In [7]:
elec_in  = flows[flows['target'] == 'Elec'].set_index('source')['realValue'].sort_values(ascending=False)
elec_out = flows[flows['source'] == 'Elec'].set_index('target')['realValue'].sort_values(ascending=False)

print('3.c  Electricity balance')
print(f'\nSupply ({elec_in.sum():,.3f} GWh)')
display(elec_in.rename('GWh').to_frame())
print(f'Uses ({elec_out.sum():,.3f} GWh)')
display(elec_out.rename('GWh').to_frame())
print(f'Residual supply - uses : {elec_in.sum() - elec_out.sum():+.6f} GWh')

3.c  Electricity balance

Supply (209.564 GWh)


,GWh
source,
Diesel gensets,184.921
Grid import (SIN),13.857
Utility PV,7.829
Home diesel gensets,2.587
Home-system PV,0.370


Uses (209.564 GWh)


,GWh
target,
Elec Demand,54.709
Food preservation,49.802
Elec Losses,32.970
Cooling,32.933
Lighting,23.186
Mechanical Energy,13.296
DEC Heat,2.669


Residual supply - uses : +0.000000 GWh


---
## 4. Sankey diagram  [manuscript figure 14]

The figure carries **no title**: the LaTeX caption does. Four editorial choices are applied
here, none of which touches the model package:

1. **Merged supply columns.** The package emits the resource row and its carrier layer as two
   nodes carrying the same quantity (`Prod. & Imp. Diesel -> Diesel` is exactly what `Diesel`
   then distributes). The three redundant links are dropped, so `Diesel`, `LPG` and `LNG` each
   become a single source node.
2. **Shared palette.** `TECH_COLORS` and `EUD_COLORS` come from `scripts/colors.py`, the
   same module `analyse_resultats.ipynb` reads, so the two chapters cannot drift apart. Nodes
   with no key in either palette keep the layer colour the package gave them. Diesel stays
   grey; `Elec Losses` takes a lighter grey so it does not read as an end use.
3. **`Elec Demand` renamed `Other electricity`** - it is the residual electricity demand, not
   the total. `Food preservation` and `Mechanical Energy` take their shorter
   `analyse_resultats` labels so the bigger type still fits at 16 cm.
4. **Explicit layout.** Each node is placed in a column by its longest path from a source
   (every end use is forced into the last column), and ordered top-down by decreasing flow,
   which is what limits the crossings.

In [8]:
from colors import TECH_COLORS, EUD_COLORS

# merge the redundant supply columns, rename the end uses
MERGE_SUPPLY = [('Prod. & Imp. Diesel', 'Diesel'), ('LPG supply', 'LPG'), ('LNG supply', 'LNG')]
RENAME = {
    'Elec Demand':       'Other electricity',   # residual electricity demand, not the total
    'Food preservation': 'Refrigeration',       # EUD_LABELS of analyse_resultats.ipynb
    'Mechanical Energy': 'Mech. energy',
}

links = flows.copy()
for supply, carrier in MERGE_SUPPLY:
    keep = ~((links['source'] == supply) & (links['target'] == carrier))
    assert (~keep).sum() == 1, f'{supply} -> {carrier}: {(~keep).sum()} links, expected 1'
    links = links[keep]
links['source'] = links['source'].replace(RENAME)
links['target'] = links['target'].replace(RENAME)
links = links[links['realValue'].abs() > 1e-9].reset_index(drop=True)

GREY_DIESEL = '#9AA5B1'     # NEUTRAL of analyse_projections.ipynb
GREY_LOSSES = '#CDD4DA'     # lighter: losses must not read as an end use

NODE_COLORS = {
    # conversion technologies
    'Diesel gensets':      TECH_COLORS['GENSET_DIESEL'],
    'Home diesel gensets': TECH_COLORS['HS_DIESEL'],
    'Utility PV':          TECH_COLORS['PV_UTILITY'],
    'Home-system PV':      TECH_COLORS['PV_HS'],
    'Grid import (SIN)':   TECH_COLORS['SIN_IMPORT'],
    'Stoves':              TECH_COLORS['STOVE_LPG'],
    'DEC Heat':            TECH_COLORS['DEC_BOILER_GAS'],
    'Elec':                TECH_COLORS['ELECTRICITY'],
    # end uses
    'Cooking':             EUD_COLORS['COOKING'],
    'Other electricity':   EUD_COLORS['ELECTRICITY'],
    'Refrigeration':       EUD_COLORS['FOOD_PRESERVATION'],
    'Cooling':             EUD_COLORS['SPACE_COOLING'],
    'Lighting':            EUD_COLORS['LIGHTING_R_C'],
    'Mech. energy':        EUD_COLORS['MECHANICAL_ENERGY_COMM'],
    'Heat LT':             EUD_COLORS['HEAT_LOW_T_DECEN'],
    # greys, as asked
    'Diesel':              GREY_DIESEL,
    'Elec Losses':         GREY_LOSSES,
}
# anything else (LPG, LNG, Fossil Gas, Solar, Regasification) keeps its layer colour
OWN_LAYER = dict(zip(links['source'], links['layerColor']))


def node_colour(name):
    return NODE_COLORS.get(name, OWN_LAYER.get(name, '#B0C4D4'))


node_names = list(dict.fromkeys(list(links['source']) + list(links['target'])))
outflow = links.groupby('source')['realValue'].sum()
inflow = links.groupby('target')['realValue'].sum()
# SIZE drives the geometry (Plotly draws a node at max(in, out));
# VALUE is what the label shows - the flow leaving the node, as the package does,
# so 'Diesel gensets' reads 184.9 GWh of electricity, not 501.1 GWh of fuel burnt.
SIZE = {n: max(float(outflow.get(n, 0.0)), float(inflow.get(n, 0.0))) for n in node_names}
VALUE = {n: float(outflow[n]) if n in outflow.index else float(inflow[n]) for n in node_names}

successors, predecessors = {}, {}
for s, t, v in zip(links['source'], links['target'], links['realValue']):
    successors.setdefault(s, []).append((t, v))
    predecessors.setdefault(t, []).append((s, v))

depth = {n: 0 for n in node_names}
for _ in range(len(node_names)):                 # longest path from any source node
    for s, targets in successors.items():
        for t, _v in targets:
            depth[t] = max(depth[t], depth[s] + 1)
LAST = max(depth.values())
for n in node_names:                             # every end use sits in the last column
    if n not in successors:
        depth[n] = LAST

columns = {}
for n in node_names:
    columns.setdefault(depth[n], []).append(n)
for col in columns:
    columns[col].sort(key=lambda n: -SIZE[n])    # decreasing flow, top-down

# Node bodies are drawn proportionally to the fullest column; pad the rest evenly.
BODY = 0.92
span = max(sum(SIZE[n] for n in members) for members in columns.values())


def place():
    """Column x from the depth, row y by stacking the current order of each column."""
    xs, ys = {}, {}
    for col, members in columns.items():
        used = sum(SIZE[n] for n in members) / span * BODY
        pad = (1.0 - used) / (len(members) + 1)
        y = pad
        for n in members:
            height = SIZE[n] / span * BODY
            xs[n] = 0.01 + 0.98 * col / LAST
            ys[n] = y + height / 2
            y += height + pad
    return xs, ys


# The last column keeps its decreasing-flow order. Every other column is then swept
# from right to left and ordered by the barycentre of its successors, with the
# predecessors breaking ties: that is what pulls each supply chain onto one level and
# leaves the diagram essentially free of crossings.
node_x, node_y = place()
for _ in range(12):
    for col in range(LAST - 1, -1, -1):
        def sort_key(n, ys=node_y):
            after = [(ys[t], v) for t, v in successors.get(n, [])]
            before = [(ys[s], v) for s, v in predecessors.get(n, [])]
            bary_after = sum(y * v for y, v in after) / sum(v for _, v in after) if after else ys[n]
            bary_before = (sum(y * v for y, v in before) / sum(v for _, v in before)
                           if before else bary_after)
            return (bary_after, bary_before, -SIZE[n])
        columns[col].sort(key=sort_key)
        node_x, node_y = place()

ORDER = [n for col in sorted(columns) for n in columns[col]]
INDEX = {n: i for i, n in enumerate(ORDER)}
node_labels = [f'{n} ({VALUE[n]:.1f})' for n in ORDER]

SINKS = set(links['target']) - set(links['source'])
link_colours = [node_colour(t if t in SINKS else s)
                for s, t in zip(links['source'], links['target'])]

sankey_df = pd.DataFrame({
    'source': links['source'].map(INDEX).values,
    'target': links['target'].map(INDEX).values,
    'realValue': links['realValue'].values,
    'layerColor': link_colours,
    'layerUnit': links['layerUnit'].values,
})

LINK_TRANSPARENCY = 0.62
print(f'{len(ORDER)} nodes over {LAST + 1} columns, {len(sankey_df)} links')
for col in sorted(columns):
    print(f'  column {col}: ' + ', '.join(f'{n} {VALUE[n]:.1f}' for n in columns[col]))

22 nodes over 5 columns, 23 links
  column 0: LPG 176.1, LNG 2.8, Diesel 511.9, Grid import (SIN) 13.9, Solar 8.2
  column 1: Stoves 113.0, Regasification 2.7, Diesel gensets 184.9, Home diesel gensets 2.6, Utility PV 7.8, Home-system PV 0.4
  column 2: Fossil Gas 2.7, Elec 209.6
  column 3: DEC Heat 5.1
  column 4: Cooking 113.0, Other electricity 54.7, Refrigeration 49.8, Elec Losses 33.0, Cooling 32.9, Lighting 23.2, Mech. energy 17.1, Heat LT 5.1


In [9]:
def hex_to_rgba(hex_color, alpha):
    """'#RRGGBB' -> 'rgba(r,g,b,alpha)' for the Plotly link colours."""
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return f'rgba({r},{g},{b},{alpha})'


def build_sankey_figure(df, labels, width, height, font_size=FONT_SIZE):
    """Report-ready Sankey: no title, explicit column/row placement, print-size type."""
    fig = go.Figure(data=[go.Sankey(
        arrangement='snap',
        node=dict(pad=13, thickness=13,
                  line=dict(color='#444444', width=0.5),
                  label=labels,
                  x=[node_x[n] for n in ORDER], y=[node_y[n] for n in ORDER],
                  color=[node_colour(n) for n in ORDER]),
        link=dict(source=df['source'], target=df['target'], value=df['realValue'],
                  color=df['layerColor'].apply(lambda c: hex_to_rgba(c, LINK_TRANSPARENCY)),
                  hovertemplate='%{value:.1f} GWh<extra></extra>'),
        textfont=dict(size=font_size, family='Arial', color='#1A1A1A'),
    )])
    fig.update_layout(
        font=dict(size=font_size, family='Arial'),
        width=width, height=height,
        margin=dict(l=6, r=6, t=14, b=14),       # no title: the LaTeX caption carries it
        paper_bgcolor='white', plot_bgcolor='white',
    )
    return fig


fig = build_sankey_figure(sankey_df, node_labels, WIDTH_PX, HEIGHT_PX)
fig.show()

---
## Notes

* The node labels carry the total flow through the node, as produced by
  `read_and_process_data` (outgoing total for a node that both emits and receives).
* `Elec Losses` is the electricity network loss of the model
  (`loss_network[ELECTRICITY]` = 15.96 %), obtained as the difference between the
  `END_USES` electricity row and the demand declared in `reg_demands.dat`. It is not a
  residual of the diagram.
* Per-cell diagrams are available from the same run: `input2sankey_C1.csv` ...
  `input2sankey_C5.csv` are written alongside `input2sankey_Total.csv`.
* Rerunning section 1 twice in the same kernel is safe: the patch is idempotent.
